In [6]:
import numpy as np
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
from sklearn.cluster import KMeans 
from sklearn.feature_extraction.text import TfidfVectorizer 
from tabulate import tabulate 
from collections import Counter 

In [7]:
nltk.download('punkt')
nltk.download('stopwords')
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

dataset = ["I love playing football on the weekends", 
           "I enjoy hiking and camping in the mountains", 
           "I like to read books and watch movies", 
           "I prefer playing video games over sports", 
           "I love listening to music and going to concerts"]

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return tokens

processed_dataset = [preprocess(sentence) for sentence in dataset]

for i, sentence in enumerate(processed_dataset):
    print(f"Sentence {i+1}: {sentence}")

[nltk_data] Downloading package punkt to
[nltk_data]     /home/fc69356b-75f4-4e92-aba8-
[nltk_data]     8e539ea45739/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/fc69356b-75f4-4e92-aba8-
[nltk_data]     8e539ea45739/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Sentence 1: ['love', 'playing', 'football', 'weekend']
Sentence 2: ['enjoy', 'hiking', 'camping', 'mountain']
Sentence 3: ['like', 'read', 'book', 'watch', 'movie']
Sentence 4: ['prefer', 'playing', 'video', 'game', 'sport']
Sentence 5: ['love', 'listening', 'music', 'going', 'concert']


In [10]:
processed_dataset_joined = [' '.join(tokens) for tokens in processed_dataset]

vectorizer = TfidfVectorizer() 
X = vectorizer.fit_transform(processed_dataset_joined)

In [11]:
k = 2  # Define the number of clusters 
km = KMeans(n_clusters=k) 
km.fit(X) 
 
# Predict the clusters for each document 
y_pred = km.predict(X) 
 
# Display the document and its predicted cluster in a table 
table_data = [["Document", "Predicted Cluster"]] 
table_data.extend([[doc, cluster] for doc, cluster in zip(processed_dataset_joined, y_pred)]) 
print(tabulate(table_data, headers="firstrow"))

# Print top terms per cluster 
print("\nTop terms per cluster:") 
order_centroids = km.cluster_centers_.argsort()[:, ::-1] 
terms = vectorizer.get_feature_names_out() 
for i in range(k): 
    print("Cluster %d:" % i) 
    for ind in order_centroids[i, :10]: 
        print(' %s' % terms[ind]) 
    print()

Document                              Predicted Cluster
----------------------------------  -------------------
love playing football weekend                         1
enjoy hiking camping mountain                         0
like read book watch movie                            1
prefer playing video game sport                       1
love listening music going concert                    1

Top terms per cluster:
Cluster 0:
 mountain
 hiking
 enjoy
 camping
 weekend
 read
 sport
 video
 watch
 movie

Cluster 1:
 playing
 love
 football
 weekend
 video
 concert
 game
 sport
 music
 listening



/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


In [12]:
# Calculate purity 
total_samples = len(y_pred) 
cluster_label_counts = [Counter(y_pred)] 
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples 
print("Purity:", purity)

Purity: 0.8


In [13]:
!pip install gensim

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels


In [14]:
import numpy as np 
from sklearn.cluster import KMeans 
from gensim.models import Word2Vec 
from tabulate import tabulate 
from collections import Counter 

In [15]:
tokenized_dataset = [doc.split() for doc in processed_dataset_joined] 
word2vec_model = Word2Vec(sentences=tokenized_dataset, vector_size=100, 
window=5, min_count=1, workers=4)

In [16]:
X = np.array([np.mean([word2vec_model.wv[word] for word in doc.split() if word in 
word2vec_model.wv], axis=0) for doc in processed_dataset_joined])

In [17]:
k = 2  # Define the number of clusters 
km = KMeans(n_clusters=k) 
km.fit(X) 
 
# Predict the clusters for each document 
y_pred = km.predict(X) 
 
# Tabulate the document and predicted cluster 
table_data = [["Document", "Predicted Cluster"]] 
table_data.extend([[doc, cluster] for doc, cluster in zip(processed_dataset_joined, y_pred)]) 
print(tabulate(table_data, headers="firstrow"))

Document                              Predicted Cluster
----------------------------------  -------------------
love playing football weekend                         1
enjoy hiking camping mountain                         0
like read book watch movie                            1
prefer playing video game sport                       1
love listening music going concert                    1


In [18]:
# Calculate purity 
total_samples = len(y_pred) 
cluster_label_counts = [Counter(y_pred)] 
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples 
print("Purity:", purity)

Purity: 0.8
